In [13]:
import os
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from evo2 import Evo2
from torch.nn import Module
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback, ModelCheckpoint, EarlyStopping, LearningRateMonitor, TQDMProgressBar
from pytorch_lightning.loggers import CSVLogger
from datetime import datetime
import time
import json
import gc
import matplotlib.pyplot as plt
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from collections import Counter
import re
import sys
import itertools
from tqdm.notebook import tqdm
# 在训练开始前设置日志级别
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)  # 只显示警告及以上级别
logging.getLogger("StripedHyena").setLevel(logging.WARNING)
import warnings
from pytorch_lightning.utilities.warnings import PossibleUserWarning
# 设置PyTorch内存分配器参数，减少内存碎片
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# 忽略实验日志目录已存在的警告
warnings.filterwarnings(
    "ignore", 
    message="Experiment logs directory .* exists and is not empty"
)
# 忽略训练批次数少于日志间隔的警告
warnings.filterwarnings(
    "ignore", 
    message="The number of training batches .* is smaller than the logging interval", 
    category=PossibleUserWarning
)
# 忽略加载检查点时发现不匹配键的警告
warnings.filterwarnings(
    "ignore",
    message="Found keys that are not in the model state dict but in the checkpoint"
)
warnings.filterwarnings("ignore", message="You are using `torch.load` with `weights_only=False`")
warnings.filterwarnings("ignore", message="torch.get_autocast_gpu_dtype\(\) is deprecated")

In [14]:
# RNA数据集类
class RNADataset(Dataset):
    def __init__(self, file_path, RNA_type):
        """
        RNA序列数据集
        Args:
            file_path: FASTA文件路径
        """
        self.sequences = []
        self.labels = []
        # 序列长度统计信息
        self.length_stats = {
            'min_length': float('inf'),
            'max_length': 0,
            'avg_length': 0,
            'length_distribution': {},
            'length_by_class': {0: [], 1: []}
        }
        # 读取FASTA文件
        self._read_fasta(file_path, RNA_type)
        
        # 检查是否成功提取了标签
        if len(self.labels) == 0:
            raise ValueError(f"未能从文件 {file_path} 中提取任何标签，请检查文件格式")

        # 分析并打印序列长度统计信息
        self._analyze_sequence_lengths()
    
    def _read_fasta(self, file_path, RNA_type):
        """读取FASTA格式文件，格式为 >ID|位置 后跟序列"""
        print(f"开始读取FASTA文件: {file_path}")
        
        with open(file_path, 'r') as f:
            lines = f.readlines()
        
        seq = ""
        current_label = None
        
        for i, line in enumerate(lines):
            line = line.strip()
            if line.startswith('>'):
                # 保存前一条序列
                if seq and current_label is not None:
                    self.sequences.append(seq)
                    self.labels.append(current_label)
                    seq = ""
                
                # 解析标签，格式为 >ID|位置
                header = line[1:]  # 去除>符号
                parts = header.split('|')
            
                if RNA_type == 'mi':
                    # miRNA直接读取第二个字段作为数值标签
                    if len(parts) > 1:
                        current_label = int(parts[1].strip())  # 直接转换为整数
                    else:
                        current_label = 0  # 默认值
                else:
                    # 其他RNA类型根据位置解析
                    location = parts[1].strip().lower() if len(parts) > 1 else "unknown"
                    if "cytoplasm" in location:
                        current_label = 0
                    elif "nuclear" in location or "nucleus" in location:
                        current_label = 1
                    else:
                        current_label = 0  # 未知位置设为默认
            else:
                seq += line.strip().upper().replace('U', 'T')  # U替换为T
        
        # 保存最后一条序列
        if seq and current_label is not None:
            self.sequences.append(seq)
            self.labels.append(current_label)
        
        # 打印数据集统计信息
        if self.labels:
            label_counts = {}
            for label in self.labels:
                label_counts[label] = label_counts.get(label, 0) + 1
            
            print(f"已读取 {len(self.sequences)} 条序列")
            for label, count in label_counts.items():
                print(f"类别 {label} : {count} 条序列")
        else:
            print("警告: 未能成功提取任何标签！")
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        """获取数据项 - 只返回原始序列和标签"""
        try:
            sequence = self.sequences[idx]
            label = self.labels[idx]
            
            # 返回字典，不包含特征
            return {
                'sequence': sequence,
                'label': label
            }
        except Exception as e:
            # 出错时记录并返回安全值
            if self.verbose:
                print(f"获取索引 {idx} 的数据时出错: {str(e)}")
            
            # 返回一个安全的默认值
            return {
                'sequence': "ACGT",  # 简短的默认序列
                'label': torch.tensor(0, dtype=torch.long)
            }
    
    def _analyze_sequence_lengths(self):
        """分析序列长度并打印统计信息"""
        if not self.sequences:
            return
        
        # 计算所有序列的长度
        lengths = [len(seq) for seq in self.sequences]
        
        # 基本统计
        self.length_stats['min_length'] = min(lengths)
        self.length_stats['max_length'] = max(lengths)
        self.length_stats['avg_length'] = sum(lengths) / len(lengths)
        
        # 按类别统计长度
        for i, label in enumerate(self.labels):
            self.length_stats['length_by_class'][label].append(len(self.sequences[i]))

        percentiles = [10, 25, 50, 75, 90]
        self.length_stats['percentiles'] = {
            p: int(np.percentile(lengths, p)) for p in percentiles
        }
        # 打印统计信息
        print("\n=== 序列长度统计 ===")
        print(f"最短序列: {self.length_stats['min_length']} bp")
        print(f"最长序列: {self.length_stats['max_length']} bp")
        print(f"平均长度: {self.length_stats['avg_length']:.2f} bp")
        print("\n分位数统计:")
        for p, value in self.length_stats['percentiles'].items():
            print(f"{p}% 的序列长度 ≤ {value} bp")
        
        print("\n按类别统计:")
        for label, lengths in self.length_stats['length_by_class'].items():
            if lengths:  # 确保列表不为空
                # label_name = "细胞质" if label == 0 else "细胞核"
                print(f"类别 {label} :")
                print(f"  最短: {min(lengths)} bp")
                print(f"  最长: {max(lengths)} bp")
                print(f"  平均: {sum(lengths)/len(lengths):.2f} bp")

    def get_length_stats(self):
        """返回序列长度统计信息"""
        return self.length_stats

In [15]:
# 序列预处理函数
def process_sequence(sequence, max_length):
    # 确保序列是字符串
    if not isinstance(sequence, str):
        sequence = str(sequence)  # 尝试转换为字符串
    
    # 截断过长序列
    if len(sequence) > max_length:
        return sequence[:max_length]
        
    # 填充过短序列
    elif len(sequence) < max_length:
        return sequence + 'N' * (max_length - len(sequence))
    
    # 长度刚好
    return sequence

class EVO2Model(nn.Module):
    def __init__(self, model_name='evo2_7b', layer_name='blocks.28.mlp.l3', max_length=512, device='cuda'):
        super().__init__()
        self.device = torch.device(device)
        self.max_length = max_length
        self.layer_name = layer_name
        
        print(f"Loading EVO2 model: {model_name} to device: {self.device}")
        self.evo2_model = Evo2(model_name)
        self.evo2_model.model.to(self.device)
        self.evo2_model.model.eval()
        print("EVO2 model loaded")

    def extract_features(self, sequences):
        if not isinstance(sequences, list):
            sequences = [sequences]
        
        processed_seqs = []
        for seq in sequences:
            if not isinstance(seq, str):
                seq = str(seq)
            if len(seq) > self.max_length:
                seq = seq[:self.max_length]
            elif len(seq) < self.max_length:
                seq = seq + 'N' * (self.max_length - len(seq))
            processed_seqs.append(seq)
        
        batch_features = []
        for seq in processed_seqs:
            input_ids = torch.tensor(
                self.evo2_model.tokenizer.tokenize(seq),
                dtype=torch.long,
            ).unsqueeze(0).to(self.device) 
            
            with torch.no_grad():
                _, embeddings = self.evo2_model(
                    input_ids, 
                    return_embeddings=True, 
                    layer_names=[self.layer_name]
                )

            features = embeddings[self.layer_name].float() 
            batch_features.append(features)
        
        return torch.cat(batch_features, dim=0).to(self.device)  


In [16]:
def generate_kmer_features(sequences, k=4, feature_dim=128, RNA_type='lnc'):
    """
    从RNA序列集合中生成k-mer特征
    """
    print(f"开始为{len(sequences)}个序列生成{k}-mer特征...")
    
    # 提取所有唯一的k-mers
    all_kmers = set()
    for seq in sequences:
        # 将U替换为T，过滤非ATCG字符
        clean_seq = ''.join([i if i in 'ATCG' else '' for i in seq.replace('U', 'T')])
        
        # 提取k-mers
        if len(clean_seq) >= k:
            kmers = [clean_seq[i:i+k] for i in range(len(clean_seq)-k+1)]
            all_kmers.update(kmers)
    
    # 构建映射字典
    kmers_list = sorted(list(all_kmers))
    kmers2id = {"UNK": 0}
    for i, kmer in enumerate(kmers_list):
        kmers2id[kmer] = i + 1  
    
    print(f"找到{len(kmers2id)}个唯一的{k}-mers (包括UNK)")
    
    # 创建特征
    node_features = np.zeros((len(kmers2id), feature_dim), dtype=np.float32)
    
    # 初始化特征矩阵
    np.random.seed(42) 
    node_features = np.random.normal(0, 0.1, size=(len(kmers2id), feature_dim))

    for kmer, idx in kmers2id.items():
        if kmer == "UNK":
            continue
            
        base_count = {'A': 0, 'T': 0, 'C': 0, 'G': 0}
        for base in kmer:
            if base in base_count:
                base_count[base] += 1/k 
        
        # 将组成信息映射到特征的前4个维度
        feature_vec = node_features[idx]
        feature_vec[:4] = [base_count['A'], base_count['T'], base_count['C'], base_count['G']]
        
        # 使用k-mer中位置特定信息
        for pos, base in enumerate(kmer):
            base_idx = {'A': 0, 'T': 1, 'C': 2, 'G': 3}.get(base, 0)
            offset = 4 + pos * 4 + base_idx
            if offset < feature_dim:
                feature_vec[offset] = 1.0
    
    # 对特征归一化
    norms = np.sqrt(np.sum(node_features**2, axis=1, keepdims=True))
    norms[norms == 0] = 1.0 
    node_features = node_features / norms
    
    # 保存文件
    output_dir = 'data'
    os.makedirs(output_dir, exist_ok=True)
    
    np.save(os.path.join(output_dir, f'{RNA_type}_kmers2id.npy'), kmers2id)
    np.save(os.path.join(output_dir, f'{RNA_type}_node_features.npy'), node_features)
    
    print(f"k-mer特征已保存到 {output_dir} 目录")
    print(f"- kmers2id.npy: 包含{len(kmers2id)}个k-mer映射")
    print(f"- node_features.npy: 形状为{node_features.shape}")
    
    return kmers2id, node_features

In [17]:
# Graph
class GraphModel(nn.Module):
    def __init__(self, in_dim=128, hidden_dim=64, RNA_type='lnc'):
        super(GraphModel, self).__init__()
        
        # 加载k-mer字典和节点特征
        self.kmers2id = np.load(f'data/{RNA_type}_kmers2id.npy', allow_pickle=True).item()
        node_features = np.load(f'data/{RNA_type}_node_features.npy', allow_pickle=True)
        self.register_buffer('node_features', torch.from_numpy(node_features).float())
        
        # 特征提取层
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        
        # 保存维度
        self.hidden_dim = hidden_dim
        
    def to(self, device):
        self.node_features = self.node_features.to(device)
        return super().to(device)
        
    def build_graph(self, sequences):
        # 批量构建RNA序列图
        if isinstance(sequences, str):
            sequences = [sequences]
        
        device = next(self.parameters()).device    
        graphs = []
        for rna_seq in sequences:
            try:
                # 序列预处理
                rna_seq = ''.join([i if i in 'ATCG' else '' for i in rna_seq.replace('U', 'T')])
                
                # 确保序列长度足够
                if len(rna_seq) < 4:
                    rna_seq = rna_seq + 'A' * (4 - len(rna_seq)) 
                
                # 提取k-mers和ID
                idseq = [self.kmers2id.get(rna_seq[i:i+4], 0) for i in range(len(rna_seq)-4+1)]
                
                # 节点ID重映射
                newidSeq = []
                old2new = {}
                count = 0
                for eachid in idseq:
                    if eachid not in old2new:
                        old2new[eachid] = count
                        count += 1
                    newidSeq.append(old2new[eachid])
                
                # 构建边关系
                counter_uv = Counter(list(zip(newidSeq[:-1], newidSeq[1:])))
                
                # 确保有边
                if not counter_uv:
                    # 创建自环
                    if newidSeq:
                        counter_uv = {(newidSeq[0], newidSeq[0]): 1}
                    else:
                        counter_uv = {(0, 0): 1}
                
                edge_index = torch.tensor(list(counter_uv.keys()), dtype=torch.long, device=device).t().contiguous()
                edge_weight = torch.tensor(list(counter_uv.values()), dtype=torch.float, device=device)
                
                # 获取节点特征
                node_ids = list(old2new.keys())
                if not node_ids:
                    node_ids = [0]  
                
                # 确保索引在范围内
                safe_node_ids = [min(idx, len(self.node_features)-1) for idx in node_ids]
                node_features = self.node_features[safe_node_ids].clone().detach()
                
                # 创建 PyG Data 对象
                graph = Data(
                    x=node_features,
                    edge_index=edge_index,
                    edge_weight=edge_weight
                )
                
                graphs.append(graph)
            
            except Exception as e:
                print(f"构建图时出错: {str(e)}, 使用默认图")
                # 创建默认图
                x = self.node_features[:1].clone() 
                edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device) 
                edge_weight = torch.ones(1, device=device)
                
                graphs.append(Data(x=x, edge_index=edge_index, edge_weight=edge_weight))
        
        return graphs
    
    def forward(self, graphs_data):
        # 处理批量图数据
        device = next(self.parameters()).device
        
        # 处理输入是列表的情况
        if isinstance(graphs_data, list):
            # 批量处理每个图
            batch_features = []
            
            for graph in graphs_data:
                # 确保所有张量都在正确的设备上
                x = graph.x.to(device)
                edge_index = graph.edge_index.to(device)
                edge_weight = graph.edge_weight.to(device)
                
                # 应用GCN层
                x = F.relu(self.conv1(x, edge_index, edge_weight))
                x = F.dropout(x, p=0.5, training=self.training)
                x = self.conv2(x, edge_index, edge_weight)
                
                # 全局平均池化
                graph_feature = torch.mean(x, dim=0)
                batch_features.append(graph_feature)
            
            # 堆叠所有图特征
            return torch.stack(batch_features)
        
        else:
            # 单个图的情况
            x = graph.x.to(device)
            edge_index = graph.edge_index.to(device)
            edge_weight = graph.edge_weight.to(device)
            
            # 应用GCN层
            x = F.relu(self.conv1(x, edge_index, edge_weight))
            x = F.dropout(x, p=0.5, training=self.training)
            x = self.conv2(x, edge_index, edge_weight)
            
            # 全局平均池化
            return torch.mean(x, dim=0, keepdim=True)  

In [18]:
# PseKNC
def read_dna_fasta(sequence):
    """处理单个DNA序列"""
    sequence = re.sub('[^ACGT-]', '-', sequence.upper())
    return sequence

def generate_list(k, nucleotides):
    """生成k-mer列表"""
    return ["".join(e) for e in itertools.product(nucleotides, repeat=k)]

def frequency(sequence, pattern):
    """计算模式在序列中的出现频率"""
    i, j, count = 0, 0, 0
    seq_len = len(sequence)
    pat_len = len(pattern)
    
    while i < seq_len and j < pat_len:
        if sequence[i] == pattern[j]:
            i += 1
            j += 1
            if j >= pat_len:
                count += 1
                i = i - j + 1
                j = 0
        else:
            i = i - j + 1
            j = 0
    return count

def get_phychem_property():
    """获取DNA二核苷酸物理化学属性"""
    return {
        'AA': [0.06, 0.5, 0.27, 1.59, 0.11, -0.11],
        'AC': [1.50, 0.50, 0.80, 0.13, 1.29, 1.04],
        'AG': [0.78, 0.36, 0.09, 0.68, -0.24, -0.62],
        'AT': [1.07, 0.22, 0.62, -1.02, 2.51, 1.17],
        'CA': [-1.38, -1.36, -0.27, -0.86, -0.62, -1.25],
        'CC': [0.06, 1.08, 0.09, 0.56, -0.82, 0.24],
        'CG': [-1.66, -1.22, -0.44, -0.82, -0.29, -1.39],
        'CT': [0.78, 0.36, 0.09, 0.68, -0.24, -0.62],
        'GA': [-0.08, 0.5, 0.27, 0.13, -0.39, 0.71],
        'GC': [-0.08, 0.22, 1.33, -0.35, 0.65, 1.59],
        'GG': [0.06, 1.08, 0.09, 0.56, -0.82, 0.24],
        'GT': [1.50, 0.50, 0.80, 0.13, 1.29, 1.04],
        'TA': [-1.23, -2.37, -0.44, -2.24, -1.51, -1.39],
        'TC': [-0.08, 0.5, 0.27, 0.13, -0.39, 0.71],
        'TG': [-1.38, -1.36, -0.27, -0.86, -0.62, -1.25],
        'TT': [0.06, 0.5, 0.27, 1.59, 0.11, -0.11]
    }

def parallel_cor_function(nucleotide1, nucleotide2, phyche_index):
    """计算平行相关函数"""
    temp_sum = 0.0
    phyche_index_values = list(phyche_index.values())
    len_phyche_index = len(phyche_index_values[0])
    
    for u in range(len_phyche_index):
        temp_sum += pow(
            float(phyche_index[nucleotide1][u]) - 
            float(phyche_index[nucleotide2][u]), 
            2
        )
    return temp_sum / len_phyche_index

def get_parallel_factor(lambda_, sequence, phyche_value):
    """计算θ列表"""
    theta = []
    l = len(sequence)
    
    for i in range(1, lambda_ + 1):
        temp_sum = 0.0
        for j in range(0, l - 1 - i):
            nucleotide1 = sequence[j] + sequence[j + 1]
            nucleotide2 = sequence[j + i] + sequence[j + i + 1]
            temp_sum += parallel_cor_function(nucleotide1, nucleotide2, phyche_value)
        theta.append(temp_sum / (l - i - 1))
    return theta
            
class PseKNCModel(nn.Module):
    def __init__(self, k=3, lambda_value=10, w=0.05, hidden_dim=256, output_dim=256):
        super().__init__()
        self.k = k
        self.lambda_value = lambda_value
        self.w = w
        self.nucleotides = 'ACGT'
        
        # 获取物理化学属性和k-mer列表
        self.phyche_value = get_phychem_property()
        self.kmer_list = generate_list(self.k, self.nucleotides)
        self.feature_dim = 4**k + lambda_value
        
        # 特征转换网络
        self.feature_net = nn.Sequential(
            nn.Linear(self.feature_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def _extract_pseknc_features(self, sequence):
        """提取单个序列的PseKNC特征"""
        try:
            # 预处理序列
            sequence = read_dna_fasta(sequence)
            
            # 检查序列长度
            if len(sequence) < self.k or self.lambda_value + self.k > len(sequence):
                raise ValueError(f"序列长度必须大于 {self.lambda_value + self.k}")
            
            # 计算k-mer频率
            freq_list = [frequency(sequence, key) for key in self.kmer_list]
            freq_sum = float(sum(freq_list))
            freq_list = [f / freq_sum for f in freq_list]
            
            # 计算θ列表
            theta_list = get_parallel_factor(self.lambda_value, sequence, self.phyche_value)
            
            # 计算最终特征向量
            theta_sum = sum(theta_list)
            denominator = 1 + self.w * theta_sum
            
            # 组合特征
            features = []
            features.extend([f / denominator for f in freq_list])
            features.extend([self.w * theta / denominator for theta in theta_list])
            
            return np.array(features, dtype=np.float32)
            
        except Exception as e:
            print(f"序列处理失败: {str(e)}")
            return None
    
    def forward(self, batch):
        """前向传播"""
        try:
            # 批量处理序列
            features = []
            for seq in batch['sequence']:
                feature = self._extract_pseknc_features(seq)
                if feature is not None:
                    features.append(feature)
                else:
                    features.append(np.zeros(self.feature_dim))
            
            # 转换为张量
            features = torch.tensor(np.array(features), dtype=torch.float32, device=self.device)
            
            # 特征转换
            output = self.feature_net(features)
            
            return output
            
        except Exception as e:
            print(f"PseKNC处理错误: {str(e)}")
            return torch.zeros(
                (len(batch['sequence']), self.feature_net[-1].out_features),
                device=self.device
            )
    
    @property
    def device(self):
        return next(self.parameters()).device

In [19]:
class ClassificationMetric(object): 
    def __init__(self, accuracy=True, recall=True, precision=True, f1=True, average="macro"):
        self.accuracy = accuracy
        self.recall = recall
        self.precision = precision
        self.f1 = f1
        self.average = average

        self.preds = []
        self.target = []

    def reset(self): # 重置结果
        self.preds.clear()
        self.target.clear()
        gc.collect()

    def update(self, preds, target): # 更新结果
        preds = list(preds.cpu().detach().argmax(1).numpy())
        target = list(target.cpu().detach().argmax(1).numpy()) if target.dim() > 1 else list(target.cpu().detach().numpy())
        self.preds += preds
        self.target += target

    def compute(self): # 计算结果
        metrics = []
        if self.accuracy:
            metrics.append(accuracy_score(self.target, self.preds))
        if self.recall:
            metrics.append(recall_score(self.target, self.preds, labels=list(set(self.preds)), average=self.average))
        if self.precision:
            metrics.append(precision_score(self.target, self.preds, labels=list(set(self.preds)), average=self.average))
        if self.f1:
            metrics.append(f1_score(self.target, self.preds, labels=list(set(self.preds)), average=self.average))
        self.reset()
        return metrics

In [20]:
class RNALocMV(pl.LightningModule):
    def __init__(
        self, 
        RNA_type,
        evo_dim=4096,
        graph_in_dim=128,  
        graph_dim=64,
        pseknc_dim=256,
        hidden_dim=512,
        fusion_dim=384,
        num_classes=2,
        dropout=0.3,
        learning_rate=1e-4,
        feature_weights=[0.6, 0.2, 0.2],  # EVO, Graph, PseKNC权重
        max_length=1024,  
        model_name='evo2_7b',  
        layer_name='blocks.28.mlp.l3',  
        pseknc_k=3,  
        pseknc_lambda=10, 
        pseknc_w=0.05 
    ):
        super().__init__()
        self.save_hyperparameters()
        
        rna_base_map = {
            'mi': 1.5,
            'lnc': 1.3,
            'circ': 1.1,
        }
        self.base = rna_base_map.get(RNA_type, 1.0)
        self.type = RNA_type

        # 特征提取器
        self.evo_model = EVO2Model(
            model_name=model_name,
            layer_name=layer_name,
            max_length=max_length,
            device = 'cuda'
        )
        
        self.graph_model = GraphModel(
            in_dim=graph_in_dim,  
            hidden_dim=graph_dim,  
            RNA_type=RNA_type
        )
        
        self.pseknc_model = PseKNCModel(
            k=pseknc_k,
            lambda_value=pseknc_lambda,
            w=pseknc_w,
            hidden_dim=hidden_dim,
            output_dim=pseknc_dim
        )
        
        # 特征转换层
        self.evo_reducer = nn.Sequential(
            nn.Linear(evo_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU()
        )
        
        self.graph_projector = nn.Sequential(
            nn.Linear(graph_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU()
        )
        
        self.pseknc_projector = nn.Sequential(
            nn.Linear(pseknc_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU()
        )
        
        # 特征融合层
        self.feature_weights = nn.Parameter(torch.tensor(feature_weights), requires_grad=True)
        
        # 自注意力机制
        self.attention = nn.MultiheadAttention(embed_dim=fusion_dim, num_heads=8, dropout=dropout, batch_first=True)
        self.layer_norm = nn.LayerNorm(fusion_dim)
        
        # 分类器
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        
        # 损失函数与指标
        self.criterion = nn.CrossEntropyLoss()
        self.train_metrics = ClassificationMetric()
        self.val_metrics = ClassificationMetric()
        self.test_metrics = ClassificationMetric()
        
        # 历史记录
        self.history = {
            'loss': [], 'acc': [], 'f1': [], 'precision': [], 'recall': [],
            'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_precision': [], 'val_recall': []
        }
        
        # 暂存批次输出
        self.train_batch_outputs = []
        self.val_batch_outputs = []
        self.test_batch_outputs = []

        # 收集所有测试批次的注意力权重
        self.attention_weights = []
        self.testing = False
    
    def forward(self, batch):
        """前向传播"""
        device = self.device
        
        # 提取三种特征
        evo_features = self.evo_model.extract_features(batch['sequence'])
        
        # 构建图并提取图特征
        graphs = self.graph_model.build_graph(batch['sequence'])
        graph_features = self.graph_model(graphs)
        
        # 提取PseKNC特征
        pseknc_features = self.pseknc_model(batch)
        
        # 特征降维与标准化
        evo_features = self.evo_reducer(evo_features)  # [batch, seq_len, fusion_dim]
        
        # 处理图特征 
        batch_size = evo_features.size(0)
        seq_len = evo_features.size(1)
        graph_features = self.graph_projector(graph_features)  # [batch, fusion_dim]
        graph_features = graph_features.unsqueeze(1).expand(-1, seq_len, -1)  # [batch, seq_len, fusion_dim]
        
        # 处理PseKNC特征
        pseknc_features = self.pseknc_projector(pseknc_features)  # [batch, fusion_dim]
        pseknc_features = pseknc_features.unsqueeze(1).expand(-1, seq_len, -1)  # [batch, seq_len, fusion_dim]
        
        # 特征融合
        weights = F.softmax(self.feature_weights, dim=0)
        fused_features = (
            weights[0] * evo_features + 
            weights[1] * graph_features + 
            weights[2] * pseknc_features
        )  # [batch, seq_len, fusion_dim]
        
        # 自注意力机制
        attn_output, attn_weights = self.attention(fused_features, fused_features, fused_features)
        fused_features = self.layer_norm(fused_features + attn_output)  

        # 收集attention权重
        if hasattr(self, 'collect_attention') and self.collect_attention:
            avg_attn = attn_weights.mean(dim=1).detach().cpu()
            if not hasattr(self, 'attention_weights'):
                self.attention_weights = []
            self.attention_weights.append(avg_attn)
            print(f"收集到attention权重，形状: {avg_attn.shape}") 
       
        
        # 池化
        sequence_repr = torch.mean(fused_features, dim=1)  # [batch, fusion_dim]
        
        # 分类
        logits = self.classifier(sequence_repr)
        
        return logits

    def set_class_weights(self, train_labels):
        """设置类别权重 - 使用更轻的惩罚
        """
        # 计算类别权重
        labels_array = np.array(train_labels)
        classes = np.unique(labels_array)
        
        # 计算类别分布
        class_counts = np.bincount(labels_array)
        total_samples = len(labels_array)
        
        # 打印类别信息
        print("\n=== 类别分布统计 ===")
        for i, cls in enumerate(classes):
            count = np.sum(labels_array == cls)
            percentage = count / total_samples * 100
            print(f"类别 {cls}: {count} 样本 ({percentage:.2f}%)")
        
        # 计算样本不平衡比例
        if len(class_counts) > 1:
            ratio = max(class_counts) / min(class_counts)
            majority_cls = np.argmax(class_counts)
            minority_cls = np.argmin(class_counts)
            print(f"多数类 ({majority_cls}) 与少数类 ({minority_cls}) 的比例: {ratio:.2f}")
        else:
            ratio = 1.0
            print("警告: 仅有一个类别!")
        
        # 为少数类增加适度权重
        weights = np.ones(len(classes))
        if len(classes) > 1 and ratio > self.base:  # 只在明显不平衡时应用权重  
            # 使用平方根缩放不平衡比例
            scaling_factor = np.sqrt(ratio) * self.base
            
            # 设置少数类的权重
            weights[minority_cls] = scaling_factor
            
            # 限制权重的最大值
            max_weight = 2.5
            weights = np.minimum(weights, max_weight)
        
        # 转换为张量并移动到适当的设备
        device = self.device
        class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
        
        # 更新损失函数
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        
        # 打印权重信息
        print("\n=== 类别权重（轻量惩罚）===")
        for i, cls in enumerate(classes):
            print(f"类别 {cls}: 权重 = {weights[i]:.4f}")
        print("=" * 50)
        
        return class_weights
    
    def training_step(self, batch, batch_idx):
        """训练步骤"""
        logits = self(batch)
        loss = self.criterion(logits, batch['label'])
        
        self.train_batch_outputs.append({
            'loss': loss.detach(),
            'logits': logits.detach(),
            'labels': batch['label'].detach()
        })
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        """验证步骤"""
        logits = self(batch)
        loss = self.criterion(logits, batch['label'])
        
        self.val_batch_outputs.append({
            'loss': loss.detach(),
            'logits': logits.detach(),
            'labels': batch['label'].detach()
        })

        # 记录准确率的分布情况
        preds = torch.argmax(logits, dim=1)
        if batch_idx == 0:
            label_counts = torch.bincount(batch['label'], minlength=self.hparams.num_classes)
            pred_counts = torch.bincount(preds, minlength=self.hparams.num_classes)
            print(f"val样本标签分布: {label_counts.cpu().numpy()}")
            print(f"val预测分布: {pred_counts.cpu().numpy()}")
        
        return loss
    
    def test_step(self, batch, batch_idx):
        """测试步骤"""
        logits = self(batch)
        loss = self.criterion(logits, batch['label'])
        
        self.test_batch_outputs.append({
            'loss': loss.detach(),
            'logits': logits.detach(),
            'labels': batch['label'].detach()
        })

        # 记录准确率的分布情况
        preds = torch.argmax(logits, dim=1)
        if batch_idx == 0:
            label_counts = torch.bincount(batch['label'], minlength=self.hparams.num_classes)
            pred_counts = torch.bincount(preds, minlength=self.hparams.num_classes)
            print(f"test样本标签分布: {label_counts.cpu().numpy()}")
            print(f"test预测分布: {pred_counts.cpu().numpy()}")
        
        return loss
    
    def on_train_epoch_end(self):
        """训练轮结束计算指标"""
        # 计算平均损失
        if not self.train_batch_outputs:
            return
            
        train_loss = torch.stack([x['loss'] for x in self.train_batch_outputs]).mean()
        
        # 收集所有预测和标签
        all_logits = torch.cat([x['logits'] for x in self.train_batch_outputs])
        all_labels = torch.cat([x['labels'] for x in self.train_batch_outputs])
        
        # 更新指标计算器
        self.train_metrics.update(all_logits, all_labels)
        
        # 计算指标
        metrics = self.train_metrics.compute()
        train_acc, train_recall, train_prec, train_f1 = metrics
        
        # 更新历史记录
        self.history['loss'].append(train_loss.item())
        self.history['acc'].append(train_acc)
        self.history['recall'].append(train_recall)
        self.history['precision'].append(train_prec)
        self.history['f1'].append(train_f1)
        
        # 记录指标
        self.log('train_loss', train_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_acc', train_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_f1', train_f1, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_precision', train_prec, on_step=False, on_epoch=True)
        self.log('train_recall', train_recall, on_step=False, on_epoch=True)
        
        # 清空批次输出
        self.train_batch_outputs.clear()
    
    def on_validation_epoch_end(self):
        """验证轮结束计算指标"""
        # 检查是否有验证批次输出
        if not self.val_batch_outputs:
            return
            
        # 计算平均验证损失
        val_loss = torch.stack([x['loss'] for x in self.val_batch_outputs]).mean()
        
        # 收集所有预测和标签
        all_logits = torch.cat([x['logits'] for x in self.val_batch_outputs])
        all_labels = torch.cat([x['labels'] for x in self.val_batch_outputs])
        
        # 更新指标计算器
        self.val_metrics.update(all_logits, all_labels)
        
        # 计算指标
        metrics = self.val_metrics.compute()
        val_acc, val_recall, val_prec, val_f1 = metrics
        
        # 更新历史记录
        self.history['val_loss'].append(val_loss.item())
        self.history['val_acc'].append(val_acc)
        self.history['val_recall'].append(val_recall)
        self.history['val_precision'].append(val_prec)
        self.history['val_f1'].append(val_f1)
        
        # 记录指标
        self.log('val_loss', val_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_acc', val_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_f1', val_f1, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_precision', val_prec, on_step=False, on_epoch=True)
        self.log('val_recall', val_recall, on_step=False, on_epoch=True)
        
        # 打印当前验证结果
        current_epoch = self.current_epoch
        print(f"Epoch {current_epoch} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
        
        # 清空批次输出
        self.val_batch_outputs.clear()
        
        self.val_metrics.reset()

    def on_test_start(self):
        """测试开始时初始化"""
        self.testing = True
        self.test_attention_weights = []
    
    def save_attention_weights(self, save_dir, filename_prefix='test'):
        """保存attention权重为CSV文件"""
        # 检查各种可能的属性名
        attention_data = []
        
        if hasattr(self, 'attention_weights') and self.attention_weights:
            attention_data = self.attention_weights
            print(f"找到attention_weights，包含 {len(attention_data)} 个batch")
        elif hasattr(self, 'test_attention_weights') and self.test_attention_weights:
            attention_data = self.test_attention_weights
            print(f"找到test_attention_weights，包含 {len(attention_data)} 个batch")
        else:
            print(f"没有收集到{filename_prefix}注意力权重")
            return None
        
        try:
            os.makedirs(save_dir, exist_ok=True)
            
            all_attn = torch.cat(attention_data, dim=0)
            print(f"{filename_prefix} attention权重形状: {all_attn.shape}")
            
            # 处理不同维度的attention权重
            all_attn_np = all_attn.numpy()
            
            if len(all_attn_np.shape) == 3:
                num_samples, seq_len, _ = all_attn_np.shape
                flat_attn = all_attn_np.reshape(num_samples, -1)
                columns = [f'attn_{i}_{j}' for i in range(seq_len) for j in range(seq_len)]
            elif len(all_attn_np.shape) == 2:
                num_samples, num_features = all_attn_np.shape
                flat_attn = all_attn_np
                import math
                seq_len = int(math.sqrt(num_features))
                if seq_len * seq_len == num_features:
                    columns = [f'attn_{i}_{j}' for i in range(seq_len) for j in range(seq_len)]
                else:
                    columns = [f'attn_feat_{i}' for i in range(num_features)]
            else:
                raise ValueError(f"不支持的attention权重维度: {all_attn_np.shape}")
            
            # 创建DataFrame
            import pandas as pd
            df = pd.DataFrame(flat_attn, columns=columns)
            df.index.name = 'sample_id'
            
            # 保存CSV
            csv_path = os.path.join(save_dir, f'{filename_prefix}_attention_weights.csv')
            df.to_csv(csv_path, index=True)
            
            print(f"{filename_prefix}的注意力权重已保存到: {csv_path}")
            
            # 清空内存
            if hasattr(self, 'attention_weights'):
                self.attention_weights = []
            if hasattr(self, 'test_attention_weights'):
                self.test_attention_weights = []
            
            return csv_path
        except Exception as e:
            print(f"保存{filename_prefix}注意力权重时出错: {str(e)}")
            return None

    def on_test_end(self):
        """测试结束时自动保存注意力权重"""
        if hasattr(self, 'logger') and self.logger and hasattr(self.logger, 'log_dir') and self.logger.log_dir:
            save_dir = self.logger.log_dir
        else:
            save_dir = 'logs/current_version'
        
        self.save_attention_weights(save_dir, 'test')
        self.testing = False
        
    def on_test_epoch_end(self):
        """测试轮结束计算指标"""
        # 检查是否有测试批次输出
        if not self.test_batch_outputs:
            return
            
        # 计算平均测试损失
        test_loss = torch.stack([x['loss'] for x in self.test_batch_outputs]).mean()
        
        # 收集所有预测和标签
        all_logits = torch.cat([x['logits'] for x in self.test_batch_outputs])
        all_labels = torch.cat([x['labels'] for x in self.test_batch_outputs])
        
        # 计算预测类别
        all_preds = torch.argmax(all_logits, dim=1)
        
        # 更新指标计算器
        self.test_metrics.update(all_logits, all_labels)
        
        # 计算指标
        metrics = self.test_metrics.compute()
        test_acc, test_recall, test_prec, test_f1 = metrics
        
        # 计算混淆矩阵
        confusion = torch.zeros(self.hparams.num_classes, self.hparams.num_classes)
        for t, p in zip(all_labels.cpu().view(-1), all_preds.cpu().view(-1)):
            confusion[t.long(), p.long()] += 1
            
        # 记录指标
        self.log('test_loss', test_loss, on_step=False, on_epoch=True)
        self.log('test_acc', test_acc, on_step=False, on_epoch=True)
        self.log('test_f1', test_f1, on_step=False, on_epoch=True)
        self.log('test_precision', test_prec, on_step=False, on_epoch=True)
        self.log('test_recall', test_recall, on_step=False, on_epoch=True)
        
        # 创建详细的测试结果字典供返回
        test_results = {
            'test_loss': test_loss.item(),
            'test_acc': test_acc,
            'test_f1': test_f1,
            'test_precision': test_prec,
            'test_recall': test_recall,
            'confusion_matrix': confusion.tolist()
        }
        
        # 计算每个类别的准确率
        class_acc = {}
        for i in range(self.hparams.num_classes):
            class_total = confusion[i, :].sum().item()
            if class_total > 0:
                class_acc[i] = confusion[i, i].item() / class_total
            else:
                class_acc[i] = 0
        
        test_results['class_accuracy'] = class_acc
        
        # 打印详细测试结果
        print("\n=== 测试结果 ===")
        print(f"测试损失: {test_loss:.4f}")
        print(f"测试准确率: {test_acc:.4f}")
        print(f"测试F1分数: {test_f1:.4f}")
        print(f"测试精确率: {test_prec:.4f}")
        print(f"测试召回率: {test_recall:.4f}")
        
        # 打印混淆矩阵
        print("\n混淆矩阵:")
        print(confusion)
        
        # 打印每个类别的准确率
        print("\n各类别准确率:")
        for cls, acc in class_acc.items():
            label_name = "细胞质" if cls == 0 else "细胞核"
            print(f"类别 {cls} ({label_name}): {acc:.4f}")
        
        # 获取特征重要性
        importance = self.get_feature_importance()
        print("\n特征重要性:")
        print(f"EVO2特征: {importance['evo']:.4f}")
        print(f"图特征: {importance['graph']:.4f}")
        print(f"PseKNC特征: {importance['pseknc']:.4f}")
        
        # 清空批次输出
        self.test_batch_outputs.clear()
        
        # 重置指标计算器
        self.test_metrics.reset()
        
        # 将测试结果添加到模型，便于外部访问
        self.test_results = test_results
        
        return test_results
    
    def configure_optimizers(self):
        """配置优化器"""
        if self.type == 'mi':
            # 分离基础层和分类器层参数
            base_params = []
            classifier_params = []
            
            # 收集各模块参数
            for name, param in self.named_parameters():
                if 'classifier' in name or 'attention' in name:
                    classifier_params.append(param)
                else:
                    base_params.append(param)
            
            # miRNA 采用分层学习率 + 余弦退火
            optimizer = torch.optim.AdamW([
                {'params': base_params, 'lr': self.hparams.learning_rate/10},
                {'params': classifier_params, 'lr': self.hparams.learning_rate}
            ], weight_decay=0.01)
            
            scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, 
                T_0=10,    
                T_mult=2,   
                eta_min=1e-6 
            )
        else:
            optimizer = torch.optim.AdamW(
                self.parameters(), 
                lr=self.hparams.learning_rate,
                weight_decay=0.01
            )
    
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode='max',
                factor=0.5,
                patience=5
            )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_acc",
                "interval": "epoch",
                "frequency": 1
            }
        }
    
    def get_feature_importance(self):
        """获取特征重要性权重"""
        weights = F.softmax(self.feature_weights, dim=0)
        return {
            "evo": weights[0].item(),
            "graph": weights[1].item(),
            "pseknc": weights[2].item()
        }
        
    @property
    def device(self):
        return next(self.parameters()).device

## train

In [21]:
def train(config): 
    print("开始训练")
    os.makedirs(config['log_dir'], exist_ok=True)
    pl.seed_everything(config['random_seed'])

    print(f"\n加载训练数据集: {config['train_data_path']}")
    train_full_dataset = RNADataset(
        file_path=config['train_data_path'],
        RNA_type=config['RNA_type']
    )
    print(f"训练数据集加载完成，共 {len(train_full_dataset)} 个样本")
    train_stats = train_full_dataset.get_length_stats()
    print(f"序列长度范围: {train_stats['min_length']} - {train_stats['max_length']} bp")
    
    print(f"\n加载测试数据集: {config['test_data_path']}")
    test_dataset = RNADataset(
        file_path=config['test_data_path'],
        RNA_type=config['RNA_type']
    )
    print(f"测试数据集加载完成，共 {len(test_dataset)} 个样本")
    test_stats = test_dataset.get_length_stats()
    print(f"序列长度范围: {test_stats['min_length']} - {test_stats['max_length']} bp")
    
    # 将训练集划分为训练集和验证集
    train_labels = train_full_dataset.labels
    train_indices, val_indices = train_test_split(
        np.arange(len(train_full_dataset)),
        test_size=config['val_split'],
        stratify=train_labels,
        random_state=config['random_seed']
    )
    
    # 创建数据子集
    train_dataset = Subset(train_full_dataset, train_indices)
    val_dataset = Subset(train_full_dataset, val_indices)
    
    # 打印数据集划分情况
    print("\n=== 数据集划分 ===")
    print(f"训练集: {len(train_indices)} 样本")
    print(f"验证集: {len(val_indices)} 样本")
    print(f"测试集: {len(test_dataset)} 样本")
    
    # 创建数据加载器
    train_loader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=config['num_workers'],
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=config['num_workers'],
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=config['num_workers'],
        pin_memory=True
    )

    type = config['RNA_type']
    # 生成kmer特征文件
    if not os.path.exists(f'data/{type}_kmers2id.npy') or not os.path.exists(f'data/{type}_node_features.npy'):
        print("\n=== 从训练集生成k-mer特征 ===")
        
        # 收集训练集序列
        train_sequences = []
        for idx in train_indices:
            train_sequences.append(train_full_dataset[idx]['sequence'])
        
        print(f"使用 {len(train_sequences)} 个训练序列生成特征...")
        
        # 生成特征
        kmers2id, node_features = generate_kmer_features(
            sequences=train_sequences, 
            k=4, 
            feature_dim=config['graph_in_dim'], 
            RNA_type=type
        )
        
        print("k-mer特征生成完成")
    else:
        print("\nk-mer特征文件已存在，跳过生成")

    # 创建PyTorch Lightning日志记录器
    logger = CSVLogger(
        save_dir=config['log_dir'],
        name=f"RNALocMV/{config['RNA_type']}RNA",  
        version=None  
    )
    
    # 获取实际的版本日志目录
    version_log_dir = logger.log_dir
    print(f"\n日志将保存在: {version_log_dir}")
    
    # 在版本目录下创建checkpoints子目录
    checkpoints_dir = os.path.join(version_log_dir, 'checkpoints')
    os.makedirs(checkpoints_dir, exist_ok=True)
    
    # 初始化模型
    model = RNALocMV(
        RNA_type=type,
        evo_dim=config['evo_dim'],
        graph_in_dim=config.get('graph_in_dim', 128), 
        graph_dim=config['graph_dim'],
        pseknc_dim=config['pseknc_dim'],
        hidden_dim=config['hidden_dim'],
        fusion_dim=config['fusion_dim'],
        num_classes=config['num_classes'],
        
        # 训练参数
        dropout=config['dropout'],
        learning_rate=config['learning_rate'],
        feature_weights=config['feature_weights'],
        
        # EVO2模型参数
        max_length=config.get('max_length', 1024),
        model_name=config.get('model_name', 'evo2_7b'),
        layer_name=config.get('layer_name', 'blocks.28.mlp.l3'),
        
        # PseKNC参数
        pseknc_k=config.get('pseknc_k', 3),
        pseknc_lambda=config.get('pseknc_lambda', 10),
        pseknc_w=config.get('pseknc_w', 0.05)
    )
    
    # 设置类别权重
    train_subset_labels = [train_full_dataset.labels[i] for i in train_indices]
    model.set_class_weights(train_subset_labels)
    
    # 创建回调
    callbacks = [
        TQDMProgressBar(refresh_rate=1),
        ModelCheckpoint(
            dirpath=checkpoints_dir,
            filename="best_acc-{epoch:02d}-{val_acc:.3f}",
            monitor="val_acc",
            mode="max",
            save_top_k=1, 
            verbose=True
        ),
        ModelCheckpoint(
            dirpath=checkpoints_dir,
            filename="best_f1-{epoch:02d}-{val_f1:.3f}",
            monitor="val_f1",
            mode="max", 
            save_top_k=1, # 3
            verbose=True
        ),
        LearningRateMonitor(logging_interval='epoch'),
    ]
    
    # 创建PyTorch Lightning训练器
    trainer = pl.Trainer(
        max_epochs=config['max_epochs'],
        accelerator='gpu',  
        devices=1,
        callbacks=callbacks,
        logger=logger,
        log_every_n_steps=50,
        enable_progress_bar=True,
        enable_model_summary=True,
        enable_checkpointing=True,
        check_val_every_n_epoch=1,
        gradient_clip_val=config['gradient_clip_val']
    )
    
    # 开始训练
    print(f"\n=== 训练开始于 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ===")
    trainer.fit(model, train_loader, val_loader)
    
    # 提取训练结果
    best_acc_model = [c for c in callbacks if isinstance(c, ModelCheckpoint) and c.monitor == 'val_acc'][0]
    best_f1_model = [c for c in callbacks if isinstance(c, ModelCheckpoint) and c.monitor == 'val_f1'][0]
    
    # 在测试集上评估最佳模型
    print("\n=== 在测试集上评估模型 ===")
    torch.cuda.empty_cache()
    
    # 加载最佳准确率模型
    best_acc_model_path = best_acc_model.best_model_path
    print(f"加载best_acc模型: {best_acc_model_path}")
    best_acc_model_loaded = RNALocMV.load_from_checkpoint(best_acc_model_path, strict=False)
    
    # 在测试集上评估
    test_results_acc = trainer.test(best_acc_model_loaded, test_loader)[0]
    print(f"best_acc模型在测试集上的结果: {test_results_acc}")

    best_acc_feature_importance = best_acc_model_loaded.get_feature_importance()

    # 删除第一个模型，释放内存
    del best_acc_model_loaded
    torch.cuda.empty_cache()

    # 加载最佳F1模型
    best_f1_model_path = best_f1_model.best_model_path
    print(f"加载best_f1模型: {best_f1_model_path}")
    best_f1_model_loaded = RNALocMV.load_from_checkpoint(best_f1_model_path, strict=False)
    
    # 在测试集上评估
    test_results_f1 = trainer.test(best_f1_model_loaded, test_loader)[0]
    print(f"best_f1模型在测试集上的结果: {test_results_f1}")
    
    # 保存配置文件
    with open(os.path.join(version_log_dir, 'config.json'), 'w') as f:
        json.dump(config, f, indent=2)
    
    # 汇总结果
    results = {
        'training': {
            'best_val_acc': best_acc_model.best_model_score.item() if hasattr(best_acc_model, 'best_model_score') else None,
            'best_val_f1': best_f1_model.best_model_score.item() if hasattr(best_f1_model, 'best_model_score') else None,
            'final_val_acc': model.history['val_acc'][-1] if model.history['val_acc'] else None,
            'final_val_f1': model.history['val_f1'][-1] if model.history['val_f1'] else None,
            'total_epochs': trainer.current_epoch + 1
        },
        'testing': {
            'best_acc_model': test_results_acc,
            'best_f1_model': test_results_f1
        },
        'paths': {
            'log_dir': version_log_dir,
            'dataset_dir': os.path.dirname(config['train_data_path']),
            'best_acc_model_path': best_acc_model_path,
            'best_f1_model_path': best_f1_model_path
        },
        'feature_importance': best_acc_feature_importance
    }
    
    # 保存详细结果到JSON
    results_path = os.path.join(version_log_dir, 'full_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    # 输出结果摘要
    print(f"\n=== 训练结束于 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ===")
    print(
        f"[bold green]训练总结[/bold green]\n"
        f"最佳验证准确率: {results['training']['best_val_acc']:.4f}\n"
        f"最佳验证F1分数: {results['training']['best_val_f1']:.4f}\n"
        f"测试集准确率 (最佳acc模型): {test_results_acc.get('test_acc', 0):.4f}\n"
        f"测试集F1分数 (最佳acc模型): {test_results_acc.get('test_f1', 0):.4f}\n"
        f"测试集准确率 (最佳F1模型): {test_results_f1.get('test_acc', 0):.4f}\n"
        f"测试集F1分数 (最佳F1模型): {test_results_f1.get('test_f1', 0):.4f}\n"
        f"总训练轮数: {results['training']['total_epochs']}\n"
        f"详细结果保存在: {results_path}"
    )
    
    return results 

## lncRNA

In [10]:
def main():
    """主函数"""
    # 创建配置字典
    config = {
        'RNA_type': 'lnc',
        # 数据路径
        'train_data_path': 'data/data_split/lnc_train.fasta',
        'test_data_path': 'data/data_split/lnc_test.fasta',
        'log_dir': 'logs',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        
        # 特征维度
        'evo_dim': 4096,
        'graph_in_dim': 128, 
        'graph_dim': 64, 
        'pseknc_dim': 256,
        'hidden_dim': 512,
        'fusion_dim': 1024, 
        'num_classes': 2,  
        
        # 训练参数
        'dropout': 0.6, 
        'learning_rate': 1e-4,
        'feature_weights': [0.6, 0.2, 0.2],  
        
        # 数据加载参数
        'batch_size': 32,
        'num_workers': 4,
        'val_split': 0.1,
        
        # 训练控制参数
        'max_epochs': 40,
        'patience': 10,
        'gradient_clip_val': 1.0,
        'random_seed': 42,
        
        # EVO2模型参数
        'max_length': 2048,
        'model_name': 'evo2_7b',
        'layer_name': 'blocks.28.mlp.l3',
        
        # PseKNC参数
        'pseknc_k': 3,
        'pseknc_lambda': 10,
        'pseknc_w': 0.05
    }

    # 确保日志目录存在
    os.makedirs(config['log_dir'], exist_ok=True)
    
    try:
        # 训练模型
        results = train(config)
        # 创建特征重要性可视化
        try:
            labels = ['EVO2', 'Graph', 'PseKNC']
            importance = results['feature_importance']
            sizes = [importance['evo'], importance['graph'], importance['pseknc']]
            colors = ['#ff9999','#66b3ff','#99ff99']
            
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
            ax.axis('equal')
            plt.title('feature_importance')
            
            # 保存图表
            plt.savefig(os.path.join(results['paths']['log_dir'], 'feature_importance.png'))
            plt.close()
            
            print(f"特征重要性图表已保存至: {os.path.join(results['paths']['log_dir'], 'feature_importance.png')}")
            
        except Exception as e:
            print(f"创建可视化分析时出错: {str(e)}")
        
    except Exception as e:
        import traceback

        error_log_path = os.path.join(config['log_dir'], 'error.log')
        with open(error_log_path, 'w') as f:
            f.write(traceback.format_exc())
        
        print(f"训练过程中发生错误: {str(e)}")
        print(f"错误详情已保存到: {error_log_path}")

In [ ]:
if __name__ == "__main__":
    main()

## circRNA

In [12]:
def main():
    """主函数"""
    # 创建配置字典
    config = {
        'RNA_type': 'circ',
        # 数据路径
        'train_data_path': 'data/data_split/circ_train.fasta',
        'test_data_path': 'data/data_split/circ_test.fasta',
        'log_dir': 'logs',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        
        # 特征维度
        'evo_dim': 4096,
        'graph_in_dim': 128,
        'graph_dim': 64,
        'pseknc_dim': 256,
        'hidden_dim': 512,
        'fusion_dim': 1024, 
        'num_classes': 2, 
        
        # 训练参数
        'dropout': 0.6,
        'learning_rate': 1e-4,
        'feature_weights': [0.6, 0.2, 0.2],  
        
        # 数据加载参数
        'batch_size': 32,
        'num_workers': 4,
        'val_split': 0.1,
        
        # 训练控制参数
        'max_epochs': 40,
        'patience': 10,
        'gradient_clip_val': 1.0,
        'random_seed': 42,
        
        # EVO2模型参数
        'max_length': 2048,
        'model_name': 'evo2_7b',
        'layer_name': 'blocks.28.mlp.l3',
        
        # PseKNC参数
        'pseknc_k': 3,
        'pseknc_lambda': 10,
        'pseknc_w': 0.05
    }

    os.makedirs(config['log_dir'], exist_ok=True)
    
    try:
        # 训练模型
        results = train(config)
        # 创建特征重要性可视化
        try:
            labels = ['EVO2', 'Graph', 'PseKNC']
            importance = results['feature_importance']
            sizes = [importance['evo'], importance['graph'], importance['pseknc']]
            colors = ['#ff9999','#66b3ff','#99ff99']
            
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
            ax.axis('equal')
            plt.title('feature_importance')
            
            # 保存图表
            plt.savefig(os.path.join(results['paths']['log_dir'], 'feature_importance.png'))
            plt.close()
            
            print(f"特征重要性图表已保存至: {os.path.join(results['paths']['log_dir'], 'feature_importance.png')}")
            
        except Exception as e:
            print(f"创建可视化分析时出错: {str(e)}")
        
    except Exception as e:
        import traceback
        
        error_log_path = os.path.join(config['log_dir'], 'error.log')
        with open(error_log_path, 'w') as f:
            f.write(traceback.format_exc())
        
        print(f"训练过程中发生错误: {str(e)}")
        print(f"错误详情已保存到: {error_log_path}")

In [ ]:
if __name__ == "__main__":
    main()

## miRNA

In [10]:
def main():
    """主函数"""
    # 创建配置字典
    config = {
        'RNA_type': 'mi',
        # 数据路径
        'train_data_path': 'data/data_split/mi_train.fasta',
        'test_data_path': 'data/data_split/mi_test.fasta',
        'log_dir': 'logs',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        
        # 特征维度
        'evo_dim': 4096,
        'graph_in_dim': 128, 
        'graph_dim': 64, 
        'pseknc_dim': 256, 
        'hidden_dim': 64,
        'fusion_dim': 512, 
        'num_classes': 2, 
        
        # 训练参数
        'dropout': 0.6,
        'learning_rate': 1e-4,
        'feature_weights': [0.6, 0.2, 0.2],  
        
        # 数据加载参数
        'batch_size': 256,
        'num_workers': 4,
        'val_split': 0.1,
        
        # 训练控制参数
        'max_epochs': 30,
        'patience': 10,
        'gradient_clip_val': 1.0,
        'random_seed': 42,
        
        # EVO2模型参数
        'max_length': 145,
        'model_name': 'evo2_7b',
        'layer_name': 'blocks.28.mlp.l3',
        
        # PseKNC参数
        'pseknc_k': 3,
        'pseknc_lambda': 10,
        'pseknc_w': 0.05
    }

    # 确保日志目录存在
    os.makedirs(config['log_dir'], exist_ok=True)
    
    try:
        # 训练模型
        results = train(config)
        # 创建特征重要性可视化
        try:
            labels = ['EVO2', 'Graph', 'PseKNC']
            importance = results['feature_importance']
            sizes = [importance['evo'], importance['graph'], importance['pseknc']]
            colors = ['#ff9999','#66b3ff','#99ff99']
            
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
            ax.axis('equal')
            plt.title('feature_importance')
            
            # 保存图表
            plt.savefig(os.path.join(results['paths']['log_dir'], 'feature_importance.png'))
            plt.close()
            
            print(f"特征重要性图表已保存至: {os.path.join(results['paths']['log_dir'], 'feature_importance.png')}")
            
        except Exception as e:
            print(f"创建可视化分析时出错: {str(e)}")
        
    except Exception as e:
        import traceback
        
        error_log_path = os.path.join(config['log_dir'], 'error.log')
        with open(error_log_path, 'w') as f:
            f.write(traceback.format_exc())
        
        print(f"训练过程中发生错误: {str(e)}")
        print(f"错误详情已保存到: {error_log_path}")

In [ ]:
if __name__ == "__main__":
    main()

## Prediction

In [22]:
def manual_test(checkpoint_path, test_dataset, save_dir, rna_type, batch_size=32, device='cuda'):
    """
    手动加载模型进行测试
    """
    import torch
    import os
    import pandas as pd
    import numpy as np
    from torch.utils.data import DataLoader
    
    # 确保保存目录存在
    os.makedirs(save_dir, exist_ok=True)
    
    # 加载模型 - 指定必需的参数
    print(f"加载模型: {checkpoint_path}")
    try:
        model = RNALocMV.load_from_checkpoint(
            checkpoint_path,
            RNA_type=rna_type,  
            strict=False 
        )
    except Exception as e:
        print(f"使用严格模式加载失败，尝试宽松模式: {e}")
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        
        hparams = checkpoint.get('hyper_parameters', {})
        hparams['RNA_type'] = rna_type 
        
        model = RNALocMV(**hparams)
        model.load_state_dict(checkpoint['state_dict'], strict=False)
    
    model.eval()
    model = model.to(device)

    # 创建测试数据加载器
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )
    
    print("开始测试...")
    all_logits = []
    all_labels = []
    all_sample_ids = []
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            for key in batch:
                if isinstance(batch[key], torch.Tensor):
                    batch[key] = batch[key].to(device)
            
            # 前向传播
            logits = model(batch)
            
            # 收集预测结果
            all_logits.append(logits.cpu())
            all_labels.append(batch['label'].cpu())
            
            # 收集样本ID
            if 'sample_id' in batch:
                all_sample_ids.extend(batch['sample_id'])
            else:
                batch_size_actual = batch['label'].size(0)
                start_idx = batch_idx * batch_size
                all_sample_ids.extend([f"sample_{start_idx + i}" for i in range(batch_size_actual)])
            
            if batch_idx % 10 == 0:
                print(f"测试进度: {batch_idx}/{len(test_loader)}")
    
    # 计算测试指标
    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    all_preds = torch.argmax(all_logits, dim=1)
    
    # 计算预测概率
    all_probs = torch.softmax(all_logits, dim=1)
    
    # 计算准确率
    accuracy = (all_preds == all_labels).float().mean()
    
    # 计算其他指标
    from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
    f1 = f1_score(all_labels.numpy(), all_preds.numpy(), average='macro')
    precision = precision_score(all_labels.numpy(), all_preds.numpy(), average='macro')
    recall = recall_score(all_labels.numpy(), all_preds.numpy(), average='macro')
    
    print(f"\n=== 测试结果 ===")
    print(f"准确率: {accuracy:.4f}")
    print(f"F1分数: {f1:.4f}")
    print(f"精确率: {precision:.4f}")
    print(f"召回率: {recall:.4f}")
    
    # 打印详细分类报告
    print("\n=== 详细分类报告 ===")
    print(classification_report(all_labels.numpy(), all_preds.numpy()))
    
    # 6. 保存预测结果
    print("\n保存预测结果...")
    
    # 创建预测结果DataFrame
    predictions_df = pd.DataFrame({
        'sample_id': all_sample_ids,
        'true_label': all_labels.numpy(),
        'predicted_label': all_preds.numpy(),
        'is_correct': (all_preds == all_labels).numpy()
    })
    
    # 添加每个类别的预测概率
    num_classes = all_probs.size(1)
    for i in range(num_classes):
        predictions_df[f'prob_class_{i}'] = all_probs[:, i].numpy()
    
    # 添加最高预测概率
    predictions_df['max_prob'] = torch.max(all_probs, dim=1)[0].numpy()
    
    # 保存预测结果
    predictions_path = os.path.join(save_dir, f'{rna_type}_test_predictions.csv')
    predictions_df.to_csv(predictions_path, index=False)
    print(f"预测结果已保存到: {predictions_path}")
    
    # 保存预测概率矩阵
    probs_path = os.path.join(save_dir, f'{rna_type}_test_probabilities.npy')
    np.save(probs_path, all_probs.numpy())
    print(f"预测概率矩阵已保存到: {probs_path}")
    
    # 保存logits
    logits_path = os.path.join(save_dir, f'{rna_type}_test_logits.npy')
    np.save(logits_path, all_logits.numpy())
    print(f"预测logits已保存到: {logits_path}")
    
    # 保存测试指标
    metrics_dict = {
        'accuracy': accuracy.item(),
        'f1_macro': f1,
        'precision_macro': precision,
        'recall_macro': recall,
        'total_samples': len(all_labels),
        'correct_predictions': int((all_preds == all_labels).sum()),
        'rna_type': rna_type
    }
    
    metrics_path = os.path.join(save_dir, f'{rna_type}_test_metrics.csv')
    metrics_df = pd.DataFrame([metrics_dict])
    metrics_df.to_csv(metrics_path, index=False)
    print(f"测试指标已保存到: {metrics_path}")

    print("\n保存错误分析...")
    wrong_predictions = predictions_df[predictions_df['is_correct'] == False]
    if len(wrong_predictions) > 0:
        wrong_predictions_path = os.path.join(save_dir, f'{rna_type}_wrong_predictions.csv')
        wrong_predictions.to_csv(wrong_predictions_path, index=False)
        print(f"错误预测样本已保存到: {wrong_predictions_path}")
        print(f"总共 {len(wrong_predictions)} 个错误预测")
    else:
        print("没有错误预测！")
    
    # 10. 返回结果
    results = {
        'accuracy': accuracy.item(),
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'predictions': all_preds.numpy(),
        'labels': all_labels.numpy(),
        'logits': all_logits.numpy(),
        'probabilities': all_probs.numpy(),
        'predictions_df': predictions_df,
        'save_paths': {
            'predictions': predictions_path,
            'probabilities': probs_path,
            'logits': logits_path,
            'metrics': metrics_path
        }
    }
    
    if len(wrong_predictions) > 0:
        results['save_paths']['wrong_predictions'] = wrong_predictions_path
    
    print(f"\n=== 所有文件保存完成 ===")
    print(f"保存目录: {save_dir}")
    
    return results

In [ ]:
# 加载测试数据集
test_dataset = RNADataset(
    file_path='data/data_split/lnc_test.fasta',
    RNA_type='lnc'
)

# 指定模型checkpoint路径
checkpoint_path = "logs/RNALocMV/lncRNA/version_6/checkpoints/best_acc-epoch=23-val_acc=0.681.ckpt"
save_dir = 'prediction'

results = manual_test(
    checkpoint_path=checkpoint_path,
    test_dataset=test_dataset,
    save_dir=save_dir,
    rna_type='lnc', 
    batch_size=1,
    device='cuda'
)

print(f"\n已保存")

In [ ]:
# 加载测试数据集
test_dataset = RNADataset(
    file_path='data/data_split/mi_test.fasta',
    RNA_type='mi'
)

# 指定模型checkpoint路径
checkpoint_path = "logs/RNALocMV/miRNA/version_1/checkpoints/best_acc-epoch=00-val_acc=0.896.ckpt"
save_dir = 'prediction'

results = manual_test(
    checkpoint_path=checkpoint_path,
    test_dataset=test_dataset,
    save_dir=save_dir,
    rna_type='mi', 
    batch_size=32,
    device='cuda'
)


print(f"\n已保存")

In [ ]:
# 加载测试数据集
test_dataset = RNADataset(
    file_path='data/data_split/circ_test.fasta',
    RNA_type='circ'
)

# 指定模型checkpoint路径
checkpoint_path = "logs/RNALocMV/circRNA/version_0/checkpoints/best_acc-epoch=38-val_acc=0.823.ckpt"
save_dir = 'prediction'

results = manual_test(
    checkpoint_path=checkpoint_path,
    test_dataset=test_dataset,
    save_dir=save_dir,
    rna_type='circ', 
    batch_size=32,
    device='cuda'
)

print(f"\n已保存")